In [4]:
import os
import sys
import pyspark

print("Python:", sys.version)
print("PySpark:", pyspark.__version__)
print("JAVA_HOME:", os.environ.get("JAVA_HOME"))

Python: 3.14.2 (main, Oct 28 2026, 19:03:49) [GCC 13.3.0]
PySpark: 3.5.6
JAVA_HOME: /usr/lib/jvm/java-17-openjdk-amd64


In [3]:
import os

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PATH"] = (
    os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]
)

print(os.environ["JAVA_HOME"])

/usr/lib/jvm/java-17-openjdk-amd64


In [5]:
from pyspark.sql import SparkSession    

# Initialize Spark session with Iceberg configurations
spark = SparkSession.builder \
  .appName("IcebergLocalDevelopment") \
  .config('spark.jars.packages', 'org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.2') \
  .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
  .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog") \
  .config("spark.sql.catalog.local.type", "hadoop") \
  .config("spark.sql.catalog.local.warehouse", "spark-warehouse/iceberg") \
  .getOrCreate()

# Verify Spark session creation
spark.sql("SHOW DATABASES").show()

26/11/24 07:43:09 WARN Utils: Your hostname, codespaces-c9dfa0 resolves to a loopback address: 127.0.0.1; using 10.0.10.7 instead (on interface eth0)
26/11/24 07:43:09 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/workspaces/learn_icebergy/iceberg_pyspark_project/.venv/lib/python3.14/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/codespace/.ivy2/cache
The jars for the packages stored in: /home/codespace/.ivy2/jars
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-552f9446-b79a-46c4-bac3-7d75ba1a858b;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.5.2 in central
:: resolution report :: resolve 230ms :: artifacts dl 7ms
	:: modules in use:
	org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.5.2 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   1   |   0   |   0   |   0   ||   1   |   0   |
	---------------------------------------------------------------------
:: retrieving :: org.apa

+---------+
|namespace|
+---------+
|  default|
+---------+



In [2]:
import os
import sys
import pyspark

print("Python:", sys.version)
print("PySpark:", pyspark.__version__)
print("JAVA_HOME:", os.environ.get("JAVA_HOME"))

Python: 3.14.2 (main, Oct 28 2026, 19:03:49) [GCC 13.3.0]
PySpark: 3.5.6
JAVA_HOME: /usr/local/sdkman/candidates/java/current


In [6]:
# Read data from the sample file
df = spark.read.csv("/workspaces/learn_icebergy/iceberg_pyspark_project/customers-5L.csv", header=True, inferSchema=True)

In [7]:
# check columns
df.columns

['Index',
 'Customer Id',
 'First Name',
 'Last Name',
 'Company',
 'City',
 'Country',
 'Phone 1',
 'Phone 2',
 'Email',
 'Subscription Date',
 'Website']

In [3]:
# Rename all the columns
df = df.withColumnRenamed("Customer Id", "Customer_Id")
df = df.withColumnRenamed("First Name", "First_Name")
df = df.withColumnRenamed("Last Name", "Last_Name")
df = df.withColumnRenamed("Phone 1", "Phone_1")
df = df.withColumnRenamed("Phone 2", "Phone_2")
df = df.withColumnRenamed("Subscription Date", "Subscription_Date")

In [17]:
# Create iceberg table with limmited number of records.
df.limit(5000).writeTo("local.schema.customers_6k") \
  .using("iceberg") \
  .createOrReplace()

26/11/24 07:52:21 WARN HadoopTableOperations: Error reading version hint file spark-warehouse/iceberg/schema/customers_6k/metadata/version-hint.text
java.io.FileNotFoundException: File spark-warehouse/iceberg/schema/customers_6k/metadata/version-hint.text does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:779)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1100)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:769)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.hadoop.fs.ChecksumFileSystem$ChecksumFSInputChecker.<init>(ChecksumFileSystem.java:160)
	at org.apache.hadoop.fs.ChecksumFileSystem.open(ChecksumFileSystem.java:372)
	at org.apache.hadoop.fs.FileSystem.open(FileSystem.java:976)
	at org.apache.iceberg.hadoop.HadoopTableOperations.findVersion(HadoopTableOperations.java:318)
	at org.apache.iceberg.

In [18]:
result = spark.table("local.schema.customers_6k")

result.show(10)
print(result.count())

+-----+---------------+----------+---------+--------------------+----------------+--------------------+--------------------+--------------------+--------------------+-----------------+--------------------+
|Index|    Customer Id|First Name|Last Name|             Company|            City|             Country|             Phone 1|             Phone 2|               Email|Subscription Date|             Website|
+-----+---------------+----------+---------+--------------------+----------------+--------------------+--------------------+--------------------+--------------------+-----------------+--------------------+
|    1|e685B8690f9fbce|      Erik|   Little|     Blankenship PLC|    Caitlynmouth|Sao Tome and Prin...|        457-542-6899|   055.415.2664x5425|shanehester@campb...|       2021-12-23| https://wagner.com/|
|    2|6EDdBA3a2DFA7De|    Yvonne|     Shaw|     Jensen and Sons|       Janetfort|Palestinian Terri...|          9610730173|   531-482-3000x7085|  kleinluis@vang.com|       202

In [16]:
df.show(10)

+-----+---------------+----------+---------+--------------------+----------------+--------------------+--------------------+--------------------+--------------------+-----------------+--------------------+
|Index|    Customer Id|First Name|Last Name|             Company|            City|             Country|             Phone 1|             Phone 2|               Email|Subscription Date|             Website|
+-----+---------------+----------+---------+--------------------+----------------+--------------------+--------------------+--------------------+--------------------+-----------------+--------------------+
|    1|e685B8690f9fbce|      Erik|   Little|     Blankenship PLC|    Caitlynmouth|Sao Tome and Prin...|        457-542-6899|   055.415.2664x5425|shanehester@campb...|       2021-12-23| https://wagner.com/|
|    2|6EDdBA3a2DFA7De|    Yvonne|     Shaw|     Jensen and Sons|       Janetfort|Palestinian Terri...|          9610730173|   531-482-3000x7085|  kleinluis@vang.com|       202

In [20]:
spark.sql("SHOW CATALOGS").show(truncate=False)

spark.sql("SHOW NAMESPACES IN local").show(truncate=False)

spark.sql("SHOW TABLES IN local.schema").show(truncate=False)

+-------------+
|catalog      |
+-------------+
|local        |
|spark_catalog|
+-------------+

+---------+
|namespace|
+---------+
|schema   |
+---------+

+---------+------------+-----------+
|namespace|tableName   |isTemporary|
+---------+------------+-----------+
|schema   |customers_5k|false      |
|schema   |customers_6k|false      |
+---------+------------+-----------+



In [14]:
spark.sql("SELECT * FROM local.schema.users").show()

AnalysisException: [TABLE_OR_VIEW_NOT_FOUND] The table or view `local`.`schema`.`users` cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.
To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS.; line 1 pos 14;
'Project [*]
+- 'UnresolvedRelation [local, schema, users], [], false


In [13]:
# Select data from the created table without partition
spark.sql("select * from local.schema.customers_5k where month(Subscription_Date) = '2'").show()

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column or function parameter with name `Subscription_Date` cannot be resolved. Did you mean one of the following? [`Subscription Date`, `First Name`, `Last Name`, `Website`, `City`].; line 1 pos 52;
'Project [*]
+- 'Filter ('month('Subscription_Date) = 2)
   +- SubqueryAlias local.schema.customers_5k
      +- RelationV2[Index#170, Customer Id#171, First Name#172, Last Name#173, Company#174, City#175, Country#176, Phone 1#177, Phone 2#178, Email#179, Subscription Date#180, Website#181] local.schema.customers_5k local.schema.customers_5k


In [11]:
# Explain the plan for the above query
spark.sql("explain select * from local.schema.customers_5k where month(Subscription_Date) = '2'").explain(mode="cost")

== Optimized Logical Plan ==
CommandResult [plan#257], Execute ExplainCommand, [[== Physical Plan ==
*(1) Filter (isnotnull(Subscription_Date#272) AND (month(Subscription_Date#272) = 2))
+- *(1) ColumnarToRow
   +- BatchScan local.schema.customers_5k[Index#262, Customer_Id#263, First_Name#264, Last_Name#265, Company#266, City#267, Country#268, Phone_1#269, Phone_2#270, Email#271, Subscription_Date#272, Website#273] local.schema.customers_5k (branch=null) [filters=Subscription_Date IS NOT NULL, groupedBy=] RuntimeFilters: []

]], Statistics(sizeInBytes=28.0 B)
   +- ExplainCommand 'Project [*], SimpleMode

== Physical Plan ==
CommandResult [plan#257]
   +- Execute ExplainCommand
         +- ExplainCommand 'Project [*], SimpleMode




In [20]:
# Create iceberg table with partition
spark.sql("""
CREATE TABLE local.schema.customers_partitioned_by_date(
  Index INT,
  Customer_Id STRING,
  First_Name STRING,
  Last_Name STRING,
  Company STRING,
  City STRING,
  Country STRING,
  Phone_1 STRING,
  Phone_2 STRING,
  Email STRING,
  Subscription_Date DATE,
  Website STRING
)
USING iceberg
PARTITIONED BY (months(Subscription_Date))
LOCATION 'spark-warehouse/iceberg/schema/customers_partitioned_by_date'
TBLPROPERTIES (
  'format' = 'iceberg/parquet',
  'format-version' = '2',
  'write.parquet.compression-codec' = 'zstd'
)
""")

DataFrame[]

In [7]:
# write data to partitioned table
df.limit(5000).writeTo("local.schema.customers_partitioned_by_date").append()

In [11]:
# Execute
spark.sql("SELECT * FROM local.schema.customers_partitioned_by_date WHERE Subscription_Date = '2021'").show()


+-----+---------------+----------+---------+--------------------+-------------------+--------------------+--------------------+--------------------+--------------------+-----------------+--------------------+
|Index|    Customer_Id|First_Name|Last_Name|             Company|               City|             Country|             Phone_1|             Phone_2|               Email|Subscription_Date|             Website|
+-----+---------------+----------+---------+--------------------+-------------------+--------------------+--------------------+--------------------+--------------------+-----------------+--------------------+
|    2|6EDdBA3a2DFA7De|    Yvonne|     Shaw|     Jensen and Sons|          Janetfort|Palestinian Terri...|          9610730173|   531-482-3000x7085|  kleinluis@vang.com|       2021-01-01|https://www.paul....|
| 1405|dD7012CFFa1dbED|    Steven|   Dunlap|        Horne-Conway|  New Autumnborough|            Guernsey|    133.063.0367x635|        711.847.2122|  ehorne@henson.

In [ ]:
spark.sql("SELECT * FROM local.schema.users").show()

AnalysisException: [TABLE_OR_VIEW_NOT_FOUND] The table or view `local`.`schema`.`users` cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.
To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS.; line 1 pos 14;
'Project [*]
+- 'UnresolvedRelation [local, schema, users], [], false


In [10]:
# Explain the plan for the above query
spark.sql("explain select * from local.schema.customers_partitioned_by_date where Subscription_Date = '2021'").explain(mode="cost")

== Optimized Logical Plan ==
CommandResult [plan#382], Execute ExplainCommand, [[== Physical Plan ==
*(1) Filter (Subscription_Date#397 = 2021-01-01)
+- *(1) ColumnarToRow
   +- BatchScan local.schema.customers_partitioned_by_date[Index#387, Customer_Id#388, First_Name#389, Last_Name#390, Company#391, City#392, Country#393, Phone_1#394, Phone_2#395, Email#396, Subscription_Date#397, Website#398] local.schema.customers_partitioned_by_date (branch=null) [filters=Subscription_Date IS NOT NULL, Subscription_Date = 18628, groupedBy=] RuntimeFilters: []

]], Statistics(sizeInBytes=28.0 B)
   +- ExplainCommand 'Project [*], SimpleMode

== Physical Plan ==
CommandResult [plan#382]
   +- Execute ExplainCommand
         +- ExplainCommand 'Project [*], SimpleMode




In [39]:
# Explain the plan for the above query
spark.sql("explain select * from local.schema.customers_partitioned_by_date where month(Subscription_Date) = '02'").explain(mode="cost")

== Optimized Logical Plan ==
CommandResult [plan#1269], Execute ExplainCommand, [[== Physical Plan ==
*(1) Filter (month(Subscription_Date#1284) = 2)
+- *(1) ColumnarToRow
   +- BatchScan local.schema.customers_partitioned_by_date[Index#1274, Customer_Id#1275, First_Name#1276, Last_Name#1277, Company#1278, City#1279, Country#1280, Phone_1#1281, Phone_2#1282, Email#1283, Subscription_Date#1284, Website#1285] local.schema.customers_partitioned_by_date (branch=null) [filters=Subscription_Date IS NOT NULL, groupedBy=] RuntimeFilters: []

]], Statistics(sizeInBytes=28.0 B)
   +- ExplainCommand 'Project [*], SimpleMode

== Physical Plan ==
CommandResult [plan#1269]
   +- Execute ExplainCommand
         +- ExplainCommand 'Project [*], SimpleMode




In [40]:
spark.sql("explain select * from local.schema.customers_partitioned_by_date WHERE Subscription_Date >= '2021-02-01' AND Subscription_Date < '2021-03-01'").explain(mode="cost")


== Optimized Logical Plan ==
CommandResult [plan#1303], Execute ExplainCommand, [[== Physical Plan ==
*(1) ColumnarToRow
+- BatchScan local.schema.customers_partitioned_by_date[Index#1308, Customer_Id#1309, First_Name#1310, Last_Name#1311, Company#1312, City#1313, Country#1314, Phone_1#1315, Phone_2#1316, Email#1317, Subscription_Date#1318, Website#1319] local.schema.customers_partitioned_by_date (branch=null) [filters=Subscription_Date IS NOT NULL, Subscription_Date >= 18659, Subscription_Date < 18687, groupedBy=] RuntimeFilters: []

]], Statistics(sizeInBytes=28.0 B)
   +- ExplainCommand 'Project [*], SimpleMode

== Physical Plan ==
CommandResult [plan#1303]
   +- Execute ExplainCommand
         +- ExplainCommand 'Project [*], SimpleMode




In [30]:
spark.sql("DESCRIBE FORMATTED local.schema.customers_partitioned_by_date").show(truncate=False)


+------------------+-----------------------------------+-------+
|col_name          |data_type                          |comment|
+------------------+-----------------------------------+-------+
|Index             |int                                |NULL   |
|Customer_Id       |string                             |NULL   |
|First_Name        |string                             |NULL   |
|Last_Name         |string                             |NULL   |
|Company           |string                             |NULL   |
|City              |string                             |NULL   |
|Country           |string                             |NULL   |
|Phone_1           |string                             |NULL   |
|Phone_2           |string                             |NULL   |
|Email             |string                             |NULL   |
|Subscription_Date |date                               |NULL   |
|Website           |string                             |NULL   |
|                  |     

In [5]:
# Create iceberg table with TRUNCATE transformation
spark.sql("""
CREATE TABLE local.schema.customers_partitioned_by_name (
  Index INT,
  Customer_Id STRING,
  First_Name STRING,
  Last_Name STRING,
  Company STRING,
  City STRING,
  Country STRING,
  Phone_1 STRING,
  Phone_2 STRING,
  Email STRING,
  Subscription_Date DATE,
  Website STRING
)
USING iceberg
PARTITIONED BY (truncate(Last_Name,5))
LOCATION 'spark-warehouse/iceberg/schema/customers_partitioned_by_name'
TBLPROPERTIES (
  'format' = 'iceberg/parquet',
  'format-version' = '2',
  'write.parquet.compression-codec' = 'zstd'
)
""")

DataFrame[]

In [4]:
df.limit(200).writeTo("local.schema.customers_partitioned_by_name").append()

In [5]:
spark.sql("select * from local.schema.customers_partitioned_by_name where Last_Name = 'Abbott'").show()

+-----+---------------+----------+---------+------------+----------+--------------------+--------------------+---------------+--------------------+-----------------+------------------+
|Index|    Customer_Id|First_Name|Last_Name|     Company|      City|             Country|             Phone_1|        Phone_2|               Email|Subscription_Date|           Website|
+-----+---------------+----------+---------+------------+----------+--------------------+--------------------+---------------+--------------------+-----------------+------------------+
|  101|dda3Da93Db3fA71|    Gordon|   Abbott|Wall-Bradley|Vernonfurt|United Arab Emirates|+1-587-566-4321x2456|+1-849-626-5599|cristina14@miller...|       2020-06-23|http://butler.com/|
+-----+---------------+----------+---------+------------+----------+--------------------+--------------------+---------------+--------------------+-----------------+------------------+



In [6]:
spark.sql("explain select * from local.schema.customers_partitioned_by_name where Last_Name = 'Abbott'").explain(mode = "cost")

== Optimized Logical Plan ==
CommandResult [plan#257], Execute ExplainCommand, [[== Physical Plan ==
*(1) Filter (Last_Name#265 = Abbott)
+- *(1) ColumnarToRow
   +- BatchScan local.schema.customers_partitioned_by_name[Index#262, Customer_Id#263, First_Name#264, Last_Name#265, Company#266, City#267, Country#268, Phone_1#269, Phone_2#270, Email#271, Subscription_Date#272, Website#273] local.schema.customers_partitioned_by_name (branch=null) [filters=Last_Name IS NOT NULL, Last_Name = 'Abbott', groupedBy=] RuntimeFilters: []

]], Statistics(sizeInBytes=28.0 B)
   +- ExplainCommand 'Project [*], SimpleMode

== Physical Plan ==
CommandResult [plan#257]
   +- Execute ExplainCommand
         +- ExplainCommand 'Project [*], SimpleMode




In [4]:
spark.sql("""
CREATE TABLE local.schema.customers_partitioned_by_bucket (
  Index INT,
  Customer_Id STRING,
  First_Name STRING,
  Last_Name STRING,
  Company STRING,
  City STRING,
  Country STRING,
  Phone_1 STRING,
  Phone_2 STRING,
  Email STRING,
  Subscription_Date DATE,
  Website STRING
)
USING iceberg
PARTITIONED BY (bucket(15,Customer_Id))
LOCATION 'spark-warehouse/iceberg/schema/customers_partitioned_by_bucket'
TBLPROPERTIES (
  'format' = 'iceberg/parquet',
  'format-version' = '2',
  'write.parquet.compression-codec' = 'zstd'
)
""")

DataFrame[]

In [5]:
df.limit(500).writeTo("local.schema.customers_partitioned_by_bucket").append()

In [6]:
spark.sql("select * from local.schema.customers_partitioned_by_bucket ").show(10)

+-----+---------------+----------+---------+--------------------+-----------------+--------------+--------------------+--------------------+--------------------+-----------------+--------------------+
|Index|    Customer_Id|First_Name|Last_Name|             Company|             City|       Country|             Phone_1|             Phone_2|               Email|Subscription_Date|             Website|
+-----+---------------+----------+---------+--------------------+-----------------+--------------+--------------------+--------------------+--------------------+-----------------+--------------------+
|   14|bB7E28Aa85cbDD3|  Jonathan|    Morse|         Potter-Holt|       Mcgeeburgh|Western Sahara|    054.140.9190x178|   461-138-3043x6409|tami64@rice-mclau...|       2021-06-20|https://www.patri...|
|   28|2F1E68Ed8ecDA2a|  Stefanie|   Gibson|       Wheeler Group|       Dillonside|       Armenia|       (246)767-7316|          8183240781|   edward62@rios.net|       2021-04-03|https://www.wells

In [7]:
spark.sql("explain select * from local.schema.customers_partitioned_by_bucket where Customer_Id = 'b9Da13bedEc47de'").explain(mode = "cost")

== Optimized Logical Plan ==
CommandResult [plan#258], Execute ExplainCommand, [[== Physical Plan ==
*(1) Filter (Customer_Id#264 = b9Da13bedEc47de)
+- *(1) ColumnarToRow
   +- BatchScan local.schema.customers_partitioned_by_bucket[Index#263, Customer_Id#264, First_Name#265, Last_Name#266, Company#267, City#268, Country#269, Phone_1#270, Phone_2#271, Email#272, Subscription_Date#273, Website#274] local.schema.customers_partitioned_by_bucket (branch=null) [filters=Customer_Id IS NOT NULL, Customer_Id = 'b9Da13bedEc47de', groupedBy=] RuntimeFilters: []

]], Statistics(sizeInBytes=28.0 B)
   +- ExplainCommand 'Project [*], SimpleMode

== Physical Plan ==
CommandResult [plan#258]
   +- Execute ExplainCommand
         +- ExplainCommand 'Project [*], SimpleMode




In [8]:
spark.sql("""
CREATE TABLE local.schema.customers_partitioned_by_country_city (
  Index INT,
  Customer_Id STRING,
  First_Name STRING,
  Last_Name STRING,
  Company STRING,
  City STRING,
  Country STRING,
  Phone_1 STRING,
  Phone_2 STRING,
  Email STRING,
  Subscription_Date DATE,
  Website STRING
)
USING iceberg
PARTITIONED BY (Country)
LOCATION 'spark-warehouse/iceberg/schema/customers_partitioned_by_country_city'
TBLPROPERTIES (
  'format' = 'iceberg/parquet',
  'format-version' = '2',
  'write.parquet.compression-codec' = 'zstd'
)
""")

DataFrame[]

In [9]:
df.limit(5000).writeTo("local.schema.customers_partitioned_by_country_city").append()

In [10]:
spark.sql("ALTER TABLE local.schema.customers_partitioned_by_country_city ADD PARTITION FIELD City").show()

++
||
++
++



In [4]:
df.limit(100).writeTo("local.schema.customers_partitioned_by_country_city").append()

In [5]:
spark.sql("ALTER TABLE local.schema.customers_partitioned_by_country_city ADD PARTITION FIELD year(Subscription_Date)").show()

++
||
++
++



In [6]:
df.limit(100).writeTo("local.schema.customers_partitioned_by_country_city").append()

In [12]:
spark.sql("SELECT * FROM local.schema.customers_partitioned_by_country_city.snapshots").show(truncate=False)


+-----------------------+-------------------+-------------------+---------+--------------------------------------------------------------------------------------------------------------------------------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|committed_at           |snapshot_id        |parent_id          |operation|manifest_list                                                                                                                                     |summary                                                                                                                                                                                                                                                 

In [14]:
spark.sql("SELECT * FROM local.schema.customers_partitioned_by_country_city VERSION AS OF 396991467446470913;").show()

+-----+---------------+----------+----------+--------------------+--------------------+-------+--------------------+--------------------+--------------------+-----------------+--------------------+
|Index|    Customer_Id|First_Name| Last_Name|             Company|                City|Country|             Phone_1|             Phone_2|               Email|Subscription_Date|             Website|
+-----+---------------+----------+----------+--------------------+--------------------+-------+--------------------+--------------------+--------------------+-----------------+--------------------+
|  157|8dBcFDF5ED9D5D2|    Evelyn|   Krueger|     Kelley and Sons|     East Coreyburgh|Lesotho|   (872)960-8087x638|001-729-045-6337x...|emmacameron@sloan...|       2021-04-24|http://www.valenz...|
|  236|c0Ea931f3eBb4A5|     Jerry|     Evans|        Velez-Keller|   Port Tracyborough|Lesotho|    785.352.8872x837|+1-848-210-2482x7907|kingkaitlin@mccal...|       2021-12-18|https://www.farre...|
|  342|3fb

In [16]:
spark.sql("SELECT * FROM local.schema.customers_partitioned_by_country_city TIMESTAMP AS OF '2025-04-21 10:29:54.179';").show()

+-----+---------------+----------+----------+--------------------+--------------------+-------+--------------------+--------------------+--------------------+-----------------+--------------------+
|Index|    Customer_Id|First_Name| Last_Name|             Company|                City|Country|             Phone_1|             Phone_2|               Email|Subscription_Date|             Website|
+-----+---------------+----------+----------+--------------------+--------------------+-------+--------------------+--------------------+--------------------+-----------------+--------------------+
|  157|8dBcFDF5ED9D5D2|    Evelyn|   Krueger|     Kelley and Sons|     East Coreyburgh|Lesotho|   (872)960-8087x638|001-729-045-6337x...|emmacameron@sloan...|       2021-04-24|http://www.valenz...|
|  236|c0Ea931f3eBb4A5|     Jerry|     Evans|        Velez-Keller|   Port Tracyborough|Lesotho|    785.352.8872x837|+1-848-210-2482x7907|kingkaitlin@mccal...|       2021-12-18|https://www.farre...|
|  342|3fb

In [17]:
spark.sql("SELECT * FROM local.schema.customers_partitioned_by_country_city.history;").show()

+--------------------+-------------------+-------------------+-------------------+
|     made_current_at|        snapshot_id|          parent_id|is_current_ancestor|
+--------------------+-------------------+-------------------+-------------------+
|2025-04-21 10:29:...| 396991467446470913|               NULL|               true|
|2025-04-21 10:38:...|7386602405767638480| 396991467446470913|               true|
|2025-04-21 10:44:...|1590301241980826574|7386602405767638480|               true|
+--------------------+-------------------+-------------------+-------------------+



Iceberg schema evolution

In [4]:
spark.sql("DESCRIBE TABLE  local.schema.customers_partitioned_by_country_city").show(truncate=False)

+-----------------+------------------------+-------+
|col_name         |data_type               |comment|
+-----------------+------------------------+-------+
|Index            |int                     |NULL   |
|Customer_Id      |string                  |NULL   |
|First_Name       |string                  |NULL   |
|Last_Name        |string                  |NULL   |
|Company          |string                  |NULL   |
|City             |string                  |NULL   |
|Country          |string                  |NULL   |
|Phone_1          |string                  |NULL   |
|Phone_2          |string                  |NULL   |
|Email            |string                  |NULL   |
|Subscription_Date|date                    |NULL   |
|Website          |string                  |NULL   |
|                 |                        |       |
|# Partitioning   |                        |       |
|Part 0           |Country                 |       |
|Part 1           |City                    |  

In [ ]:
# Reorder columns in iceberg
spark.sql("ALTER TABLE local.schema.customers_partitioned_by_country_city ALTER COLUMN City AFTER Customer_Id;").show(truncate=False)

In [ ]:
# Add column
spark.sql("""
    ALTER TABLE local.schema.customers_partitioned_by_country_city
    ADD COLUMN referrer STRING AFTER Website
""")


In [ ]:
# Remove column
spark.sql("""
    ALTER TABLE local.schema.customers_partitioned_by_country_city 
    DROP COLUMN referrer
""")


In [ ]:
# Rename column
spark.sql("""
    ALTER TABLE local.schema.customers_partitioned_by_country_city 
    RENAME COLUMN Website TO Website_name
""")

In [ ]:
# Change data type
spark.sql("""
    ALTER TABLE local.schema.customers_partitioned_by_country_city
    ALTER COLUMN subscription_date SET DATA TYPE TIMESTAMP
""")

In [21]:
spark.sql("SHOW CATALOGS").show(truncate=False)

spark.sql("SHOW NAMESPACES IN local").show(truncate=False)

spark.sql("SHOW TABLES IN local.schema").show(truncate=False)

+-------------+
|catalog      |
+-------------+
|local        |
|spark_catalog|
+-------------+

+---------+
|namespace|
+---------+
|schema   |
+---------+

+---------+------------+-----------+
|namespace|tableName   |isTemporary|
+---------+------------+-----------+
|schema   |customers_5k|false      |
|schema   |customers_6k|false      |
+---------+------------+-----------+



In [22]:
# Check catalog config
for k, v in spark.sparkContext.getConf().getAll():
    if "iceberg" in k or "catalog" in k or "warehouse" in k:
        print(f"{k} = {v}")

# Check namespaces inside `local`
spark.sql("SHOW NAMESPACES IN local").show()
spark.sql("SHOW TABLES IN local.schema").show()

spark.sql.catalog.local.warehouse = spark-warehouse/iceberg
spark.sql.catalog.local.type = hadoop
spark.sql.warehouse.dir = file:/workspaces/learn_icebergy/iceberg_pyspark_project/spark-warehouse
spark.sql.catalog.local = org.apache.iceberg.spark.SparkCatalog
+---------+
|namespace|
+---------+
|   schema|
+---------+

+---------+------------+-----------+
|namespace|   tableName|isTemporary|
+---------+------------+-----------+
|   schema|customers_5k|      false|
|   schema|customers_6k|      false|
+---------+------------+-----------+



In [23]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS local.schema")

df.limit(5000).writeTo("local.schema.customers_6k") \
    .createOrReplace()

In [24]:
spark.table("local.schema.customers_6k").count()

5000

In [25]:
# Select data from the created table without partition
spark.sql("select * from local.schema.customers_6k where month(Subscription_Date) = '2'").show()

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column or function parameter with name `Subscription_Date` cannot be resolved. Did you mean one of the following? [`Subscription Date`, `First Name`, `Last Name`, `Website`, `City`].; line 1 pos 52;
'Project [*]
+- 'Filter ('month('Subscription_Date) = 2)
   +- SubqueryAlias local.schema.customers_6k
      +- RelationV2[Index#785, Customer Id#786, First Name#787, Last Name#788, Company#789, City#790, Country#791, Phone 1#792, Phone 2#793, Email#794, Subscription Date#795, Website#796] local.schema.customers_6k local.schema.customers_6k


In [26]:
spark.table("local.schema.customers_6k").printSchema()

root
 |-- Index: integer (nullable = true)
 |-- Customer Id: string (nullable = true)
 |-- First Name: string (nullable = true)
 |-- Last Name: string (nullable = true)
 |-- Company: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Phone 1: string (nullable = true)
 |-- Phone 2: string (nullable = true)
 |-- Email: string (nullable = true)
 |-- Subscription Date: date (nullable = true)
 |-- Website: string (nullable = true)



In [31]:
spark.sql("""
    SELECT *
    FROM local.schema.customers_6k

""").show()

+-----+---------------+----------+---------+--------------------+----------------+--------------------+--------------------+--------------------+--------------------+-----------------+--------------------+
|Index|    Customer Id|First Name|Last Name|             Company|            City|             Country|             Phone 1|             Phone 2|               Email|Subscription Date|             Website|
+-----+---------------+----------+---------+--------------------+----------------+--------------------+--------------------+--------------------+--------------------+-----------------+--------------------+
|    1|e685B8690f9fbce|      Erik|   Little|     Blankenship PLC|    Caitlynmouth|Sao Tome and Prin...|        457-542-6899|   055.415.2664x5425|shanehester@campb...|       2021-12-23| https://wagner.com/|
|    2|6EDdBA3a2DFA7De|    Yvonne|     Shaw|     Jensen and Sons|       Janetfort|Palestinian Terri...|          9610730173|   531-482-3000x7085|  kleinluis@vang.com|       202